In [7]:
pip install tensorflow

  Using cached tensorflow-2.22.0rc0-cp314-cp314-win_amd64.whl.metadata (4.6 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.84.0-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached keras_nightly-3.16.0.dev2026092404-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.15.1-cp314-cp314-win_amd64.whl.metadata (3.1 kB)
  Using cached ml_dtypes-0.6.0-cp314-cp314-win_amd64.whl.metadata (8.8 kB)
  Using cached namex-0.1.0-py3-none-any.whl.metadata (322 bytes)

In [8]:
import pandas as pd 
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder , StandardScaler
from sklearn.metrics import mean_squared_error ,mean_absolute_error, r2_score

In [10]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt

In [11]:
df= pd.read_csv("climate_energy.csv")
print(df.head())

             timestamp    location  temperature  energy_consumption
0  2024-01-01 00:00:00     Chennai         21.1               42.27
1  2024-01-01 01:00:00       Salem         20.8               41.64
2  2024-01-01 02:00:00     Madurai         21.7               41.88
3  2024-01-01 03:00:00  Coimbatore         19.6               43.51
4  2024-01-01 04:00:00  Coimbatore         20.9               42.29


In [12]:
print(df.info())
print(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   timestamp           2000 non-null   str    
 1   location            2000 non-null   str    
 2   temperature         2000 non-null   float64
 3   energy_consumption  2000 non-null   float64
dtypes: float64(2), str(2)
memory usage: 113.9 KB
None
       temperature  energy_consumption
count  2000.000000         2000.000000
mean     26.476500           66.403965
std       4.153513           21.389659
min      16.700000           20.560000
25%      23.200000           47.997500
50%      26.500000           65.225000
75%      29.700000           82.810000
max      36.600000          124.310000


In [13]:
df["timestamp"] =pd.to_datetime(df["timestamp"])
df["hour"] =df["timestamp"].dt.hour
df["day"] =df["timestamp"].dt.month
df.drop("timestamp", axis=1, inplace=True)
print(df.head())

     location  temperature  energy_consumption  hour  day
0     Chennai         21.1               42.27     0    1
1       Salem         20.8               41.64     1    1
2     Madurai         21.7               41.88     2    1
3  Coimbatore         19.6               43.51     3    1
4  Coimbatore         20.9               42.29     4    1


In [14]:
encoder= LabelEncoder()
df["location"] = encoder.fit_transform(df["location"])
print(df.head())

   location  temperature  energy_consumption  hour  day
0         0         21.1               42.27     0    1
1         3         20.8               41.64     1    1
2         2         21.7               41.88     2    1
3         1         19.6               43.51     3    1
4         1         20.9               42.29     4    1


In [15]:
x = df.drop("energy_consumption", axis=1)
y= df["energy_consumption"]

In [16]:
scaler= StandardScaler()
x_scaled = scaler.fit_transform(x)

In [17]:
x_train, x_test, y_train, y_test = train_test_split(
    x_scaled, y, test_size=0.2, random_state=42
)
print(x_train.shape)
print(x_test.shape)

(1600, 4)
(400, 4)


In [18]:
model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(x_train.shape[1],)))
model.add(Dense(8, activation='relu'))
model.add(Dense(1))

c:\Users\User\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:102: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

In [20]:
model.summary

<bound method Model.summary of <Sequential name=sequential, built=True>>

In [21]:
history = model.fit(
    x_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
    )

Epoch 1/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 4869.2163 - mae: 66.4425 - val_loss: 4622.9258 - val_mae: 64.3678
Epoch 2/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 4667.3291 - mae: 64.8876 - val_loss: 4324.2314 - val_mae: 62.0563
Epoch 3/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 4217.8486 - mae: 61.4111 - val_loss: 3704.5496 - val_mae: 57.1475
Epoch 4/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 3401.3970 - mae: 54.6771 - val_loss: 2707.5161 - val_mae: 48.3902
Epoch 5/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2261.5747 - mae: 43.7571 - val_loss: 1541.7834 - val_mae: 35.6721
Epoch 6/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1141.9202 - mae: 29.8208 - val_loss: 645.4429 - val_mae: 22.0599
Epoch 7/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 433.9430 - mae: 17.4331 - val_loss: 241.3406 - val_mae: 13.1169
Epoch 8/100
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 179.0795 - mae: 11.0831 - val_loss: 140.2199 - val_mae: 9.8255
Epoch 

In [22]:
loss , mae = model.evaluate(x_test, y_test)
print("Test Loss:", loss)
print("Test MAE:", mae)

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 27.5612 - mae: 4.2248  
Test Loss: 27.561237335205078
Test MAE: 4.224765300750732


In [23]:
predictions = model.predict(x_test)
print(predictions[:5])

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
[[114.05541 ]
 [ 59.479103]
 [ 83.98371 ]
 [ 72.30447 ]
 [ 82.659294]]
